# Tutorial Exercise: Build an Advanced Panel App

:::{note}
:icon: false

#### Synthesis challenge: architecture, patterns, theming, and performance

In this exercise, you will build an advanced, class-based Panel application that combines the core ideas from the full tutorial sequence.

You are expected to design your app like a small production project:

- clear separation of concerns
- reusable and composable classes
- explicit reactive dependencies
- consistent visual theme
- performance-conscious rendering

## Learning goals

By the end, you should be able to:

1. Build a multi-view app using class-based composition.
2. Apply a simple, practical design pattern choice in a Panel app.
3. Implement a coherent theme and style system.
4. Measure and improve key performance bottlenecks.

:::

---

In [ ]:
from dataclasses import dataclass

import pandas as pd
import panel as pn
import panel_material_ui as pmui
import param

pn.extension("tabulator", throttled=True)

## 1) App brief

Build an app called **"Wind Turbine Operations Console"** using the local dataset `./windturbines.parq`.

Your app should help users explore wind turbine projects, apply filters, and inspect patterns through:

- KPI cards
- an interactive table
- one or two charts
- a detail panel for selected records

### Functional requirements

- Load data from `./windturbines.parq` (no synthetic dataset).
- Support at least **three filters** (e.g., manufacturer, state, year range, capacity threshold).
- Include at least **two linked views** (e.g., table selection updates detail/chart).
- Include **one user-controlled setting** that changes behavior (not only appearance).
- Include **one persisted preference** per session (e.g., default view mode).

---

## 2) Architecture constraints (class-based)

Structure your code into focused classes. Use this as a minimum blueprint:

1. `DataRepository` - owns raw data loading/caching.
2. `FilterState` - defines filter parameters and validation.
3. `ThemeConfig` - central place for colors, spacing, and sizing.
4. `DashboardView` - composes panes/widgets for the main UI.
5. `AppController` - wires dependencies and exposes `.servable()`.

### Rules

- Avoid global mutable state.
- Prefer declarative dependencies (`pn.bind`, `param.depends`, `param.rx`) for computed outputs.
- Use imperative watchers only for true side effects (logging, notifications, persistence).
- Keep each class small and testable.

---

## 3) Simple design pattern guidance

Keep this lightweight. Use **one or two** simple patterns:

- **View + State split**: one class manages state/filters, another renders UI.
- **Observer (optional)**: a small watcher for side effects (e.g., status text, logging, preference persistence).

That is enough for this exercise. Add 2-3 lines in markdown explaining what you chose.

---

In [ ]:
# Starter scaffold (edit and extend)

class DataRepository(param.Parameterized):
    path = param.String(default="./windturbines.parq")

    def load(self) -> pd.DataFrame:
        # Good first optimization target: cache this result.
        return pd.read_parquet(self.path)


class FilterState(param.Parameterized):
    manufacturers = param.ListSelector(default=[])
    states = param.ListSelector(default=[])
    year_range = param.Range(default=(2000, 2020), bounds=(1900, 2100))
    min_capacity = param.Number(default=0, bounds=(0, None))


class DataStore(param.Parameterized):
    repository = param.ClassSelector(class_=DataRepository, default=DataRepository())
    filters = param.ClassSelector(class_=FilterState, default=FilterState())

    def __init__(self, **params):
        super().__init__(**params)
        data = self.repository.load()
        self.filters.param.manufacturers.objects = sorted(data["t_manu"].dropna().unique().tolist())
        self.filters.param.states.objects = sorted(data["t_state"].dropna().unique().tolist())
        self.filters.param.year_range.bounds = (
            int(data["p_year"].min()),
            int(data["p_year"].max()),
        )
        self.filters.year_range = self.filters.param.year_range.bounds

    @param.depends(
        "filters.manufacturers",
        "filters.states",
        "filters.year_range",
        "filters.min_capacity",
    )
    def filtered_data(self):
        df = self.repository.load()
        if self.filters.manufacturers:
            df = df[df["t_manu"].isin(self.filters.manufacturers)]
        if self.filters.states:
            df = df[df["t_state"].isin(self.filters.states)]

        start_year, end_year = self.filters.year_range
        return df[
            df["p_year"].between(start_year, end_year)
            & (df["p_cap"] >= self.filters.min_capacity)
        ]


@dataclass
class ThemeConfig:
    accent: str = "#2962FF"
    bg: str = "#F8FAFC"
    text: str = "#0F172A"
    radius: int = 10

    def render_theme_config(self):
        # Designed for pmui.Page(theme_config=...).
        return {
            "light": {
                "palette": {
                    "primary": {"main": self.accent},
                    "background": {"default": self.bg, "paper": self.bg},
                    "text": {"primary": self.text},
                },
                "shape": {"borderRadius": self.radius},
                "typography": {
                    "button": {"textTransform": "none"},
                },
            },
            "dark": {
                "palette": {
                    "primary": {"main": self.accent},
                    "background": {"default": "#0B1220", "paper": "#111827"},
                    "text": {"primary": "#E5E7EB"},
                },
                "shape": {"borderRadius": self.radius},
                "typography": {
                    "button": {"textTransform": "none"},
                },
            },
        }


class DashboardView(param.Parameterized):
    data_store = param.ClassSelector(class_=DataStore)
    theme = param.Parameter()

    def view(self):
        # TODO: Replace with a composed layout:
        # - controls panel
        # - KPI cards
        # - table + chart + detail panel
        filters = self.data_store.filters
        df = self.data_store.repository.load()

        manufacturers = pmui.MultiChoice(
            label="Manufacturers",
            options=filters.param.manufacturers.objects,
            value=filters.manufacturers,
        )
        states = pmui.MultiChoice(
            label="States",
            options=filters.param.states.objects,
            value=filters.states,
        )
        years = pmui.IntRangeSlider(
            label="Year range",
            start=int(df["p_year"].min()),
            end=int(df["p_year"].max()),
            value=filters.year_range,
        )
        min_capacity = pmui.FloatSlider(
            label="Min capacity",
            start=0,
            end=float(df["p_cap"].max()),
            value=float(filters.min_capacity),
        )

        manufacturers.link(filters, value="manufacturers")
        states.link(filters, value="states")
        years.link(filters, value="year_range")
        min_capacity.link(filters, value="min_capacity")

        columns = ["p_name", "t_state", "t_county", "p_year", "t_manu", "p_cap"]
        preview = pn.bind(lambda: self.data_store.filtered_data()[columns].head(20))

        return pn.Row(
            pn.Column(
                "## TODO: Build your Wind Turbine Operations Console",
                pn.pane.Markdown("Theme is applied via `pmui.Page(theme_config=...)` in `AppController`."),
                manufacturers,
                states,
                years,
                min_capacity,
                width=360,
            ),
            pn.widgets.Tabulator(preview, sizing_mode="stretch_both"),
            sizing_mode="stretch_both",
        )


class AppController(param.Parameterized):
    data_store = param.ClassSelector(class_=DataStore, default=DataStore())
    theme = param.Parameter(default=ThemeConfig())
    title = param.String(default="Wind Turbine Operations Console")

    def servable(self):
        view = DashboardView(data_store=self.data_store, theme=self.theme)
        content = view.view()

        if pn.state.served:
            page = pmui.Page(
                title=self.title,
                theme_config=self.theme.render_theme_config(),
            )
            page.main.append(content)
            return page

        return content


app = AppController()
app.servable()

## 4) Theming requirements

Implement a consistent theme system:

- One class/object that defines color tokens, spacing, and typography choices.
- Reuse those tokens across cards, table, charts, and controls.
- Include a **light/dark toggle** or at least two named theme presets.
- Ensure readable contrast for text and important indicators.

Bonus: make your theme swappable without rewriting view logic.

---

## 5) Performance requirements

Profile and improve responsiveness. At minimum:

1. Measure one expensive path (e.g., filtering + table update).
2. Implement at least **two optimizations**, such as:
   - caching loaded data
   - reducing recomputation (memoization or reactive dependency cleanup)
   - using `throttled=True` widgets for expensive operations
   - limiting table columns/rows rendered at once
   - deferring heavy components until needed (tabs/accordion/lazy view)
3. Re-measure and report before/after timing in markdown.

Use a small benchmark helper like this:

```python
def timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = (time.perf_counter() - t0) * 1000
    print(f"{label}: {dt:.1f} ms")
    return out
```

---

## 6) Deliverables checklist

Your submission should include:

- [ ] A working class-based Panel app with `.servable()` entrypoint.
- [ ] At least 5 focused classes with clear responsibilities.
- [ ] A simple, clear architecture split (state + view), documented.
- [ ] Cohesive theme applied throughout the app.
- [ ] Before/after performance timings with short interpretation.
- [ ] A short "architecture rationale" (5-10 lines) in markdown.

---

## 7) Stretch goals (optional)

- Add deep-linking to preserve filter state in URL params.
- Add simple role-based controls (viewer vs editor).
- Add a custom component (or wrapper) for a reusable KPI tile.
- Package your app module so it can be imported and tested.

---

## 8) Reflection prompts

Write short answers (2-4 sentences each):

1. Which architectural decision most improved maintainability?
2. Where did imperative code remain necessary, and why?
3. Which optimization gave the largest responsiveness gain?
4. What would you refactor next if this app grew 10x?
